# IBGE Shapefile Column Explorer

This notebook explores the actual column names and structure of IBGE territorial boundary shapefiles (municipios and RGI).

## Objectives:
1. Download IBGE ZIPs from official sources
2. Extract shapefiles
3. Load with Sedona geospatial library
4. Display all columns and data samples
5. Identify correct column names for mapping

## Setup: Import Libraries

In [ ]:
import io
import tempfile
import zipfile
from pathlib import Path

import requests
import geopandas as gpd
from pyspark.sql import SparkSession


## Initialize Spark with Sedona

In [ ]:
spark = SparkSession.builder \
    .appName("IBGE Column Explorer") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## Define IBGE Data Sources

In [ ]:
IBGE_URLS = {
    "municipios": (
        "https://geoftp.ibge.gov.br/organizacao_do_territorio/"
        "malhas_territoriais/malhas_municipais/municipio_2024/"
        "Brasil/BR_Municipios_2024.zip"
    ),
    "rgi": (
        "https://geoftp.ibge.gov.br/organizacao_do_territorio/"
        "malhas_territoriais/malhas_municipais/municipio_2024/"
        "Brasil/BR_RG_Imediatas_2024.zip"
    ),
}

print("Data sources defined:")
for data_type, url in IBGE_URLS.items():
    print(f"  - {data_type}: {url}")

## Helper Functions

In [ ]:
def download_and_extract_zip(data_type: str, source_url: str, staging_dir: Path) -> Path:
    """Download ZIP from IBGE and extract shapefile."""
    print(f"\nDownloading {data_type}...")
    
    response = requests.get(source_url, timeout=120)
    response.raise_for_status()
    print(f"Downloaded {len(response.content) / 1024 / 1024:.1f} MB")

    with zipfile.ZipFile(io.BytesIO(response.content)) as zipped_payload:
        # Find .shp file
        shp_members = [
            member for member in zipped_payload.namelist()
            if member.lower().endswith(".shp")
        ]
        
        if not shp_members:
            raise FileNotFoundError(f"No SHP file found in {data_type} archive")

        # Extract all files (including sidecars: .shx, .dbf, .prj)
        extract_dir = staging_dir / f"ibge_{data_type}_shp"
        zipped_payload.extractall(extract_dir)
        print(f"Extracted {len(zipped_payload.namelist())} files")
        
        return extract_dir / shp_members[0]

print("Helper functions defined")

## Load Shapefile with Sedona

In [ ]:
def load_shapefile_as_dataframe(shapefile_path: Path, spark: SparkSession):
    """Load shapefile using geopandas and convert to Spark DataFrame."""
    print(f"\n📖 Loading shapefile: {shapefile_path.name}")
    
    # Load with geopandas
    gdf = gpd.read_file(str(shapefile_path))
    
    # Convert geometry to WKT for display
    gdf['geometry_wkt'] = gdf['geometry'].apply(lambda x: x.wkt)
    
    # Drop the geometry column (not needed for this exploration)
    gdf_display = gdf.drop(columns=['geometry'])
    
    # Add geometry as WKT string back
    gdf_display['geometry'] = gdf['geometry_wkt']
    gdf_display = gdf_display.drop(columns=['geometry_wkt'])
    
    # Convert to Spark DataFrame
    df = spark.createDataFrame(gdf_display)
    
    print(f"Loaded successfully ({df.count():,} rows)")
    
    return df

print("Shapefile loader defined")

## Explore Municipios Dataset

In [ ]:
with tempfile.TemporaryDirectory(dir="/tmp") as staging_dir:
    staging_path = Path(staging_dir)
    
    print("\n" + "="*80)
    print("MUNICIPIOS DATA EXPLORATION")
    print("="*80)
    
    shp_path = download_and_extract_zip(
        "municipios",
        IBGE_URLS["municipios"],
        staging_path,
    )
    
    municipios_df = load_shapefile_as_dataframe(shp_path, spark)
    
    print(f"\nTotal records: {municipios_df.count():,}")

### Municipios: Schema

In [ ]:
print("\nSCHEMA:")
print("="*80)
municipios_df.printSchema()

### Municipios: All Columns

In [ ]:
print("\nALL COLUMNS:")
print("="*80)
for i, col in enumerate(sorted(municipios_df.columns), 1):
    print(f"  {i:2d}. {col}")

print(f"\nTotal: {len(municipios_df.columns)} columns")

### Municipios: Sample Data

In [ ]:
print("\nSAMPLE DATA (first 3 records):")
print("="*80)
municipios_df.show(3, vertical=True, truncate=False)

### Municipios: Column Categories

In [ ]:
# Organize columns by category
all_cols = municipios_df.columns
categories = {
    "Geometry": [c for c in all_cols if "geom" in c.lower()],
    "Municipality": [c for c in all_cols if "mun" in c.lower()],
    "State/UF": [c for c in all_cols if "uf" in c.lower()],
    "Region/RGI": [c for c in all_cols if "rgi" in c.lower() or ("rg" in c.lower() and "regia" not in c.lower())],
    "Geographic Region": [c for c in all_cols if "regia" in c.lower()],
    "Concurrency": [c for c in all_cols if "concu" in c.lower()],
    "Area": [c for c in all_cols if "area" in c.lower()],
}

print("\nCOLUMNS BY CATEGORY:")
print("="*80)
for category, cols in sorted(categories.items()):
    if cols:
        print(f"\n{category}:")
        for col in cols:
            print(f"  - {col}")

## Explore RGI Dataset

In [ ]:
with tempfile.TemporaryDirectory(dir="/tmp") as staging_dir:
    staging_path = Path(staging_dir)
    
    print("\n" + "="*80)
    print("RGI DATA EXPLORATION")
    print("="*80)
    
    shp_path = download_and_extract_zip(
        "rgi",
        IBGE_URLS["rgi"],
        staging_path,
    )
    
    rgi_df = load_shapefile_as_dataframe(shp_path, spark)
    
    print(f"\nTotal records: {rgi_df.count():,}")

### RGI: Schema

In [ ]:
print("\nSCHEMA:")
print("="*80)
rgi_df.printSchema()

### RGI: All Columns

In [ ]:
print("\nALL COLUMNS:")
print("="*80)
for i, col in enumerate(sorted(rgi_df.columns), 1):
    print(f"  {i:2d}. {col}")

print(f"\nTotal: {len(rgi_df.columns)} columns")

### RGI: Sample Data

In [ ]:
print("\nSAMPLE DATA (first 3 records):")
print("="*80)
rgi_df.show(3, vertical=True, truncate=False)

### RGI: Column Categories

In [ ]:
# Organize columns by category
all_cols = rgi_df.columns
categories = {
    "Geometry": [c for c in all_cols if "geom" in c.lower()],
    "Region/RGI": [c for c in all_cols if "rgi" in c.lower() or ("rg" in c.lower() and "regia" not in c.lower())],
    "State/UF": [c for c in all_cols if "uf" in c.lower()],
    "Geographic Region": [c for c in all_cols if "regia" in c.lower()],
    "Area": [c for c in all_cols if "area" in c.lower()],
}

print("\nCOLUMNS BY CATEGORY:")
print("="*80)
for category, cols in sorted(categories.items()):
    if cols:
        print(f"\n{category}:")
        for col in cols:
            print(f"  - {col}")

## Summary: Recommended Column Mapping

In [ ]:
print("\n" + "="*80)
print("RECOMMENDED COLUMN MAPPING FOR BRONZE LAYER")
print("="*80)
print("\nMUNICIPIOS:")
print("-" * 80)
municipios_mapping = {
    "nome_municipio": "NM_MUN",
    "codigo_municipio": "CD_MUN",
    "sg_uf": "SIGLA_UF",
    "geometry": "geometry",
}
for output_name, source_col in municipios_mapping.items():
    exists = source_col in municipios_df.columns
    status = "ok" if exists else "nao tem"
    print(f"  {status} {output_name:25} ← {source_col}")

print("\nRGI:")
print("-" * 80)
rgi_mapping = {
    "nome_rgi": "NM_RGI",
    "codigo_rgi": "CD_RGI",
    "sg_uf": "SIGLA_UF",
    "geometry": "geometry",
}
for output_name, source_col in rgi_mapping.items():
    exists = source_col in rgi_df.columns
    status = "ok" if exists else "nao tem"
    print(f"  {status} {output_name:25} ← {source_col}")

print("\n" + "="*80)

## Cleanup

In [ ]:
spark.stop()
print("Spark session stopped")